# Data Cleaning — DataCo Smart Supply Chain for Big Data Analysis

**Project phase:** Data Cleaning (CRISP-DM Phase 3)
**Input:** `DataCoSupplyChainDataset.csv` — 180,519 rows x 53 columns
**Analyst:** Akash More

This notebook applies the cleaning plan from the Data Understanding notebook: drop columns with no analytical
value, fix missing values and data types, standardize categorical text, and enforce the business rules the data
should satisfy (positive sales, non-negative shipping days, unique order items, no missing customer IDs).

Every drop or fix below is logged with a before/after row count, so the cleaning is auditable end to end.


## 1. Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="ISO-8859-1")
df_clean = df.copy()

print(f"Starting shape: {df_clean.shape}")
df_clean.head()

Starting shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## 2. Drop Columns With No Analytical Value

`Customer Email` and `Customer Password` are masked PII with no usable signal. `Order Zipcode`, `Customer Street`,
and `Product Description` are near-fully missing. `Product Image` is a URL, not a modeling feature.

In [2]:
cols_to_drop = [
    "Customer Email", "Customer Password", "Order Zipcode",
    "Customer Street", "Product Description", "Product Image",
]

before_shape = df_clean.shape
df_clean = df_clean.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns: {before_shape} -> {df_clean.shape}")

Dropped 6 columns: (180519, 53) -> (180519, 47)


## 3. Handle Remaining Missing Values

In [3]:
missing = df_clean.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

Customer Lname      8
Customer Zipcode    3
dtype: int64

In [4]:
# Customer Lname: missing last name doesn't invalidate the row — fill with a placeholder.
df_clean["Customer Lname"] = df_clean["Customer Lname"].fillna("Unknown")

# Customer Zipcode: small number missing — mode imputation is reasonable at this scale.
df_clean["Customer Zipcode"] = df_clean["Customer Zipcode"].fillna(df_clean["Customer Zipcode"].mode()[0])

print("Remaining missing values:")
print(df_clean.isnull().sum().sum())

Remaining missing values:
0


## 4. Fix Data Types

Dates and numeric fields sometimes load as generic objects. Getting these right now avoids silent errors in every
later step (a string "Sales" column would still let `df.describe()` run, just on the wrong columns).

In [5]:
df_clean["order date (DateOrders)"] = pd.to_datetime(df_clean["order date (DateOrders)"])
df_clean["shipping date (DateOrders)"] = pd.to_datetime(df_clean["shipping date (DateOrders)"])

df_clean["Sales"] = pd.to_numeric(df_clean["Sales"], errors="coerce")

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 47 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Type                           180519 non-null  object        
 1   Days for shipping (real)       180519 non-null  int64         
 2   Days for shipment (scheduled)  180519 non-null  int64         
 3   Benefit per order              180519 non-null  float64       
 4   Sales per customer             180519 non-null  float64       
 5   Delivery Status                180519 non-null  object        
 6   Late_delivery_risk             180519 non-null  int64         
 7   Category Id                    180519 non-null  int64         
 8   Category Name                  180519 non-null  object        
 9   Customer City                  180519 non-null  object        
 10  Customer Country               180519 non-null  object        
 11  

## 5. Standardize Categorical Text

Free-text categorical columns often carry inconsistent casing or stray whitespace (`" standard class"` vs
`"Standard Class"`), which silently inflates `nunique()` and breaks groupby/merge operations later.

In [6]:
cat_cols = df_clean.select_dtypes(include="object").columns

cardinality_before = {col: df_clean[col].nunique() for col in cat_cols}

In [7]:
df_clean["Shipping Mode"] = df_clean["Shipping Mode"].str.strip().str.title()
df_clean["Customer Segment"] = df_clean["Customer Segment"].str.strip().str.title()
df_clean["Delivery Status"] = df_clean["Delivery Status"].str.strip().str.title()
df_clean["Market"] = df_clean["Market"].str.strip().str.title()

print(df_clean["Shipping Mode"].unique())
print(df_clean["Delivery Status"].unique())
print(df_clean["Customer Segment"].unique())

['Standard Class' 'First Class' 'Second Class' 'Same Day']
['Advance Shipping' 'Late Delivery' 'Shipping On Time' 'Shipping Canceled']
['Consumer' 'Home Office' 'Corporate']


In [8]:
cardinality_after = {col: df_clean[col].nunique() for col in cat_cols}

pd.DataFrame({"before": cardinality_before, "after": cardinality_after}).query("before != after")

,before,after


## 6. Enforce Business Rules

These are constraints the data should satisfy given what the columns represent — not statistical outlier detection.
Each rule is checked, counted, and then applied, so the row-count impact of every rule is visible.

In [9]:
rule_log = []

def apply_rule(name, mask_valid, frame):
    n_before = len(frame)
    n_invalid = n_before - mask_valid.sum()
    frame = frame[mask_valid]
    rule_log.append({"rule": name, "rows_before": n_before, "rows_removed": n_invalid, "rows_after": len(frame)})
    return frame

In [10]:
# Order item quantity must be positive.
df_clean = apply_rule(
    "Order Item Quantity > 0",
    df_clean["Order Item Quantity"] > 0,
    df_clean,
)

In [11]:
# Sales must be positive — a zero or negative sale value is a data-entry error, not a real transaction.
df_clean = apply_rule(
    "Sales > 0",
    df_clean["Sales"] > 0,
    df_clean,
)

In [12]:
# Delivery days can't be negative.
df_clean = apply_rule(
    "Days for shipping (real) >= 0",
    df_clean["Days for shipping (real)"] >= 0,
    df_clean,
)
df_clean = apply_rule(
    "Days for shipment (scheduled) >= 0",
    df_clean["Days for shipment (scheduled)"] >= 0,
    df_clean,
)

In [13]:
# Customer Id is required for any customer-level analysis.
df_clean = apply_rule(
    "Customer Id not null",
    df_clean["Customer Id"].notnull(),
    df_clean,
)

In [14]:
# Order Item Id should be a unique key — drop exact duplicates if any exist.
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="Order Item Id")
rule_log.append({
    "rule": "Order Item Id unique",
    "rows_before": before,
    "rows_removed": before - len(df_clean),
    "rows_after": len(df_clean),
})

pd.DataFrame(rule_log)

,rule,rows_before,rows_removed,rows_after
0,Order Item Quantity > 0,180519,0,180519
1,Sales > 0,180519,0,180519
2,Days for shipping (real) >= 0,180519,0,180519
3,Days for shipment (scheduled) >= 0,180519,0,180519
4,Customer Id not null,180519,0,180519
5,Order Item Id unique,180519,0,180519


## 7. Profit Outliers — Flagged, Not Removed

`Order Profit Per Order` includes both large gains and large losses. An IQR scan flags the extreme values, but
they are not dropped here — the Data Understanding notebook already established that negative-profit orders are
a real business signal, and extreme values may be legitimate high-value B2B orders. Removing them without
business input would quietly bias any profitability analysis downstream.

In [15]:
Q1 = df_clean["Order Profit Per Order"].quantile(0.25)
Q3 = df_clean["Order Profit Per Order"].quantile(0.75)
IQR = Q3 - Q1
lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

extreme_profit = df_clean[
    (df_clean["Order Profit Per Order"] < lower_bound) | (df_clean["Order Profit Per Order"] > upper_bound)
]
print(f"Flagged (not removed) extreme profit rows: {len(extreme_profit)} / {len(df_clean)}")

df_clean["is_profit_outlier"] = df_clean["Order Profit Per Order"].between(lower_bound, upper_bound).eq(False)

Flagged (not removed) extreme profit rows: 18942 / 180519


## 8. Final Validation

In [16]:
print(f"Final shape: {df_clean.shape}")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")
print(f"Duplicate Order Item Id: {df_clean['Order Item Id'].duplicated().sum()}")
print(f"Rows with Sales <= 0: {(df_clean['Sales'] <= 0).sum()}")
print(f"Rows with negative shipping days: {(df_clean['Days for shipping (real)'] < 0).sum()}")

df_clean.head()

Final shape: (180519, 48)
Remaining missing values: 0
Duplicate Order Item Id: 0
Rows with Sales <= 0: 0
Rows with negative shipping days: 0


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Fname,Customer Id,Customer Lname,Customer Segment,Customer State,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,is_profit_outlier
0,DEBIT,3,4,91.250000,314.640015,Advance Shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Cally,20755,Holloway,Consumer,PR,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,False
1,TRANSFER,5,4,-249.089996,311.359985,Late Delivery,1,73,Sporting Goods,Caguas,Puerto Rico,Irene,19492,Luna,Consumer,PR,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,True
2,CASH,4,4,-247.779999,309.720001,Shipping On Time,0,73,Sporting Goods,San Jose,EE. UU.,Gillian,19491,Maldonado,Consumer,CA,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,True
3,DEBIT,3,4,22.860001,304.809998,Advance Shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,Tana,19490,Tate,Home Office,CA,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,False
4,PAYMENT,2,4,134.210007,298.250000,Advance Shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Orli,19489,Hendricks,Corporate,PR,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,False


In [17]:
df_clean.to_csv("DataCoSupplyChain_Cleaned.csv", index=False)
print("Saved: DataCoSupplyChain_Cleaned.csv")

Saved: DataCoSupplyChain_Cleaned.csv


## 9. Summary

- Started at 180,519 rows / 53 columns.
- Dropped 6 columns with no analytical value (PII, near-fully-missing, or non-feature URLs).
- Imputed the two remaining columns with residual missing values.
- Fixed date and numeric dtypes.
- Standardized text casing across key categorical columns.
- Applied 6 business-rule filters (see Section 6 table for exact row counts removed by each).
- Flagged profit outliers as a column rather than dropping them, pending a business decision.
- Exported the cleaned dataset to `DataCoSupplyChain_Cleaned.csv`, ready for EDA.
